In [95]:
import os
import sys
sys.path.append('/workspace/Retinal-vessels-segmentation/')

In [96]:
import numpy as np
from PIL import Image

In [97]:
import torch
import torch.nn as nn
from utils import *
import matplotlib.pyplot as plt
import cv2
from io import BytesIO
import torch.nn.functional as F
from transforms import get_test_patch_transforms
from sklearn.metrics import f1_score, recall_score

In [ ]:
output_dir='./output'

In [ ]:
def draw_dashed_line(img, pt1, pt2, color, thickness=1, gap=10):
    """Draw a dashed line between two points"""
    dist = ((pt1[0] - pt2[0]) ** 2 + (pt1[1] - pt2[1]) ** 2) ** 0.5
    pts = []
    for i in np.arange(0, dist, gap):
        r = i / dist
        x = int((pt1[0] * (1 - r) + pt2[0] * r) + 0.5)
        y = int((pt1[1] * (1 - r) + pt2[1] * r) + 0.5)
        pts.append((x, y))
    
    for i in range(0, len(pts) - 1, 2):
        cv2.line(img, pts[i], pts[i + 1], color, thickness)

In [ ]:
def save_png_to_new_path(input_path, root_output_path,suf=''):
    img_name = input_path.split('/')[-1].split('.')[0]
    output_path = os.path.join(root_output_path, f'{img_name}_{suf}.png')
    img = cv2.imread(input_path,1)
    cv2.imwrite(output_path, img)

In [ ]:
def swap_loc(imgs):
    def filter_key(path):
        if '_gt.' in path:
            return False
        elif '_origin.' in path:
            return False
        elif 'our_net' in path:
            return False
        else :
            pass
        return True
    tmp = [0]*3
    for path in imgs:
        if '_gt.' in path:
            tmp[1] = path
        elif '_origin.' in path:
            tmp[0] = path
        elif 'our_net' in path:
            tmp[2] = path
        else : pass
    return tmp+list(filter(filter_key, imgs))

In [ ]:
import glob
class  Drawer:
    def __init__(self, models=['dysta_net','edae_net','fr_net','gtdla','our_net','sfit_net','unet']
                 ,checkpoints_path='/workspace/Retinal-vessels-segmentation/checkpoints',
                 output_dir='./output',
                 resize_size=(1024,1024)):
        self.models = models
        self.output_dir = output_dir
        self.checkpoints_path = checkpoints_path
        self.resize_size = resize_size
        if self.output_dir and not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir, exist_ok=True)
    def infer_model(self,model, image,preprocessing_func):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = model.to(device)
        model.eval()
        with torch.inference_mode():
            if isinstance(image, str):
                image = np.array(Image.open(image))
            preprocessed_image = preprocessing_func(image)
            img_tensor = get_test_patch_transforms()(image=preprocessed_image)['image'].to(device)
            _, original_h, original_w = img_tensor.shape

            
            img_tensor = mirror_padding_v2(img_tensor).unsqueeze(0)
            B, C, H, W = img_tensor.shape
            num_patch = ((H-64)//32+1, (W-64)//8+1)
            image_patches, tmp_stride = extract_patches_with_target_count(img_tensor, 64, num_patch)
            if len(image_patches.shape) > 4:
                image_patches = image_patches.flatten(0, 1)
            chunk_size = max(image_patches.shape[0] // 128, 1)
            chunk_image = torch.chunk(image_patches, chunk_size, 0)

            out_sample = []
            for c_image in chunk_image:
                with torch.inference_mode():
                    prob = model(c_image)
                out_sample.append(prob)
            prob = torch.cat(out_sample, 0)
            prob = prob.view(B, -1, 1, 64, 64)
            prob = reverse_to_original_image(prob, (H, W), 64, tmp_stride).squeeze()[:original_h, :original_w]

            # Threshold => binary mask (numpy)
            pred_mask = (prob >= 0.487).to(torch.uint8).detach().cpu().numpy()  # shape (h,w), 0/1
            display_mask = (pred_mask * 255).astype(np.uint8)
        return display_mask
    def create_zoom_inset(self,path,out_path,pos=(243, 419, 48, 30),resize_size=(1024,1024), line_style='dashed'):
        """
        line_style options:
        - 'solid': solid lines
        - 'dashed': dashed lines
        - 'both': connects all 4 corners
        - 'diagonal': connects diagonal corners
        """

        img = cv2.imread(path,1)
        h, w = img.shape[:2]

        # 1. Xác định vị trí vùng muốn cắt (x, y, width, height)
        crop_x, crop_y, crop_w, crop_h = pos 
        
        # 2. Cắt vùng đó ra
        roi = img[crop_y:crop_y+crop_h, crop_x:crop_x+crop_w]
        
        # 3. Phóng to vùng đã cắt
        zoom_scale = max(float(h)*0.2,float(w)*0.2)
        zoomed_roi = cv2.resize(roi, None, fx=zoom_scale/min(crop_w, crop_h), fy=zoom_scale/min(crop_w, crop_h), interpolation=cv2.INTER_LANCZOS4)

        # 4. Vẽ khung vàng cho vùng cắt trên ảnh gốc
        cv2.rectangle(img, (crop_x, crop_y), (crop_x+crop_w, crop_y+crop_h), (0, 255, 0), 2)

        # 5. Xác định vị trí đặt ảnh đã zoom
        zh, zw = zoomed_roi.shape[:2]
        pos_x, pos_y = w - zw, h - zh
        
        # Vẽ khung vàng cho ảnh zoom
        cv2.rectangle(zoomed_roi, (0, 0), (zw-1, zh-1), (0, 255, 0), 8)

        # 6. Đè ảnh zoom lên ảnh gốc
        img[pos_y:pos_y+zh, pos_x:pos_x+zw] = zoomed_roi

        # 7. VẼ CÁC ĐƯỜNG NỐI
        line_color = (0, 255, 255)  # Yellow (BGR)
        line_thickness = 5
        
        # Define corner points of ROI (original region)
        roi_corners = {
            'top_left': (crop_x, crop_y),
            'top_right': (crop_x + crop_w, crop_y),
            'bottom_left': (crop_x, crop_y + crop_h),
            'bottom_right': (crop_x + crop_w, crop_y + crop_h)
        }
        
        # Define corner points of zoomed inset
        zoom_corners = {
            'top_left': (pos_x, pos_y),
            'top_right': (pos_x + zw, pos_y),
            'bottom_left': (pos_x, pos_y + zh),
            'bottom_right': (pos_x + zw, pos_y + zh)
        }
        
        if line_style == 'solid':
            # Connect corresponding corners with solid lines
            cv2.line(img, roi_corners['top_right'], zoom_corners['top_left'], line_color, line_thickness)
            cv2.line(img, roi_corners['bottom_right'], zoom_corners['bottom_left'], line_color, line_thickness)
        
        elif line_style == 'dashed':
            # Connect corresponding corners with dashed lines
            draw_dashed_line(img, roi_corners['top_right'], zoom_corners['top_left'], line_color, line_thickness)
            draw_dashed_line(img, roi_corners['bottom_right'], zoom_corners['bottom_left'], line_color, line_thickness)
        
        elif line_style == 'both':
            # Connect all 4 corners
            draw_dashed_line(img, roi_corners['top_left'], zoom_corners['top_left'], line_color, line_thickness)
            draw_dashed_line(img, roi_corners['top_right'], zoom_corners['top_right'], line_color, line_thickness)
            draw_dashed_line(img, roi_corners['bottom_left'], zoom_corners['bottom_left'], line_color, line_thickness)
            draw_dashed_line(img, roi_corners['bottom_right'], zoom_corners['bottom_right'], line_color, line_thickness)
        
        elif line_style == 'diagonal':
            # Connect diagonal corners
            draw_dashed_line(img, roi_corners['top_left'], zoom_corners['bottom_right'], line_color, line_thickness)
            draw_dashed_line(img, roi_corners['bottom_right'], zoom_corners['top_left'], line_color, line_thickness)
        img = cv2.resize(img, resize_size, interpolation=cv2.INTER_LANCZOS4)
        cv2.imwrite(out_path, img)
    def concatenate_images_simple(self,images, padding=1, bg_color=(0, 0, 0),type_cat='vertical'):
        images = [np.array(cv2.imread(images[i],1)) for i in range(len(images))]
        
        if padding > 0:
            h, w, c = images[0].shape
            padding_strip = np.full((h, padding, c), bg_color, dtype=np.uint8)
            
            result = []
            for i, img in enumerate(images):
                result.append(img)
                if i < len(images) - 1:
                    result.append(padding_strip)
            if type_cat=='vertical':
                return np.vstack(result)
            elif type_cat=='horizontal':
                return np.hstack(result)
        else:
            if type_cat=='vertical':
                return np.vstack(images)
            elif type_cat=='horizontal':
                return np.hstack(images)
    def run(self, input_path,gt_path,suffix='drive',only_vs=True,pos=None):
    
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(os.path.join(self.output_dir,suffix), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir,suffix,'zoom_inset'), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir,suffix,'infer'), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir,suffix,'cat'), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir,suffix,'pos'), exist_ok=True)
        save_png_to_new_path(input_path, os.path.join(self.output_dir,suffix,'infer'),suf='origin')
        save_png_to_new_path(gt_path, os.path.join(self.output_dir,suffix,'infer'),suf='gt')
        gt = np.ceil(cv2.imread(gt_path,0).astype(np.float32)/255).astype(np.uint8)
        recalls= []
        f1_scores = []
        for model_name in self.models:
            from load_model import load_model_class
            load_model_class(model_name)
            model = torch.load(
                    os.path.join(self.checkpoints_path, f'{model_name}_{suffix}.pt'),
                    map_location='cuda' if torch.cuda.is_available() else 'cpu',
                    weights_only=False
            )
            dis = self.infer_model(model, input_path,preprocessing_img if 'our_net' not in model_name else lambda x: cv2.cvtColor(x, cv2.COLOR_RGB2GRAY))
            dis_0_1 = np.ceil(dis.astype(np.float32)/255).astype(np.uint8)
            recall = recall_score(gt.flatten(), dis_0_1.flatten())
            f1 = f1_score(gt.flatten(), dis_0_1.flatten())
            recalls.append(recall)
            f1_scores.append(f1)
            save_path = os.path.join(self.output_dir,suffix,'infer',f'{input_path.split("/")[-1].split(".")[0]}_{model_name}.png')
            cv2.imwrite(save_path, dis)
        imgs = glob.glob(os.path.join(self.output_dir,suffix,'infer','*.png'))
        imgs = swap_loc(imgs)
        if only_vs:
            fig,ax = plt.subplots(1,len(imgs)-2,figsize=(10,5))
            our_net_path = imgs[2]
            our_net_img = np.ceil(cv2.imread(our_net_path,0).astype(np.float32)/255).astype(np.uint8)
            intersec=our_net_img*gt
            fn_if = np.ones_like(gt)
            for i in range(3,len(imgs)):
                if_img = np.ceil(cv2.imread(imgs[i],0).astype(np.float32)/255).astype(np.uint8)
                intersec_dis = (np.clip(intersec - (if_img*gt),0,1)*255).astype(np.uint8)
                fn_if*=(intersec_dis//255).astype(np.uint8)
                ax[i-3].imshow(intersec_dis,cmap='gray')
                ax[i-3].set_title(f'Our net vs {imgs[i].split("_")[-2].split(".")[0]}')
                ax[i-3].axis('off')
            fn_if = np.full((fn_if.shape[0],fn_if.shape[1],3),1)*fn_if.reshape((fn_if.shape[0],fn_if.shape[1],1))*np.array([0,0,255]).astype(np.uint8)
            cv2.imwrite(os.path.join(self.output_dir,suffix,'pos',f'{input_path.split("/")[-1].split(".")[0]}_fn_if.png'),fn_if)
            ax[-1].imshow(fn_if)
            ax[-1].set_title(f'lack of intersection')
            ax[-1].axis('off')
            plt.tight_layout()
            plt.show()
            print('Recalls:', self.models[np.argmax(recalls)])
            print('F1-scores:', self.models[np.argmax(f1_scores)])
        else:
            if pos is not None:
                for path in imgs:
                    self.create_zoom_inset(path,os.path.join(self.output_dir,suffix,'zoom_inset', 'zoom_'+path.split("/")[-1]),pos=pos,line_style='dashed')
                cat_h_img = self.concatenate_images_simple(glob.glob(os.path.join(self.output_dir,suffix,'zoom_inset','*.png')), padding=0, bg_color=(0, 0, 0),type_cat='horizontal')
                cat_v_img = self.concatenate_images_simple(glob.glob(os.path.join(self.output_dir,suffix,'zoom_inset','*.png')), padding=0, bg_color=(0, 0, 0),type_cat='vertical')
                save_path_zoom = os.path.join(self.output_dir,suffix,'cat','h_cat.png')
                save_path_zoom_2 = os.path.join(self.output_dir,suffix,'cat','v_cat.png')
                cv2.imwrite(save_path_zoom, cat_h_img)
                Image.open(save_path_zoom).save(
                    save_path_zoom,
                    dpi=(300, 300)
                )
                cv2.imwrite(save_path_zoom_2, cat_v_img)
                Image.open(save_path_zoom_2).save(
                    save_path_zoom_2,
                    dpi=(300, 300)
                )
        # else:

In [ ]:
drawer = Drawer(models=['dysta_net','edae_net','fr_net','gtdla','our_net','sfit_net','unet'],
                 checkpoints_path='/workspace/Retinal-vessels-segmentation/checkpoints',
                 output_dir='./output',
                 resize_size=(1024,1024))


In [ ]:
drawer.run('/workspace/Retinal-vessels-segmentation/data/DRIVE/test/images/03_test.tif',
           '/workspace/Retinal-vessels-segmentation/data/DRIVE/test/mask/03_manual1.gif',
           suffix='drive',
           only_vs=True,)